In [4]:
import os
import pickle

import torch
import torch.nn as nn

In [ ]:
dir = "/mnt/data/activation_cache"
filepath = os.path.join(dir, "activations_00000_00100.pkl")
with open(filepath, mode="br") as f:
    ret = pickle.load(f)

NameError: name 'filepath' is not defined

In [13]:
class GPT2Activations(torch.utils.data.Dataset):
    def __init__(self, dir):
        super().__init__()
        self.dir = dir

    def __len__(self):
        pass
    
    def __get_filename(self, id):
        start = id // 100 
        end = start + 100
        path_name = f"activations_{start:05d}_{end:05d}.pkl"
        return path_name

    def __getitem__(self, id):
        filepath = os.path.join(self.dir, self.__get_filename(id))
        with open(filepath, mode="br") as f:
            data = pickle.load(f)
        return data[id % 100]

In [14]:
dataset = GPT2Activations(dir)

In [19]:
dataset[0]

{'transformer.h.0.mlp.c_fc': tensor([[[ 0.0508,  0.2176,  0.1028,  ..., -2.8691, -1.1017, -0.1803],
          [ 0.2784, -1.4891, -0.9972,  ..., -2.2001, -1.6619,  0.3665],
          [ 0.1374, -0.6176, -1.8192,  ..., -2.2956, -0.8707,  0.0229],
          ...,
          [ 0.3432, -2.1425, -1.8013,  ..., -1.3432, -1.4794, -0.4806],
          [ 0.3455, -0.5672,  0.3466,  ..., -1.0349, -0.8576, -1.6546],
          [ 0.2695, -2.1647, -1.3896,  ..., -1.4854, -1.4596,  0.0604]]]),
 'transformer.h.0.mlp.c_proj': tensor([[[-1.1143e-01,  2.3145e-01,  1.9578e-01,  ..., -2.3698e-03,
           -1.1194e+00, -5.1384e-01],
          [-5.4829e-01, -1.3611e-01, -3.2587e-01,  ..., -7.8699e-02,
            1.7837e-01,  1.1413e-01],
          [ 2.5987e+00, -2.2147e-01, -6.8006e-01,  ..., -3.0500e-01,
            2.3650e-02, -1.8889e+00],
          ...,
          [-1.0233e-01,  2.8909e-01,  4.5879e-02,  ...,  7.5369e-02,
           -5.6041e-01,  2.8636e-01],
          [-5.0780e-01,  6.0486e-01,  6.6675e-01,

In [ ]:
ret

In [ ]:
type(ret)

In [ ]:
ret[0]['transformer.h.0.mlp.c_fc'].shape

In [ ]:
ret[0]['transformer.h.0.mlp.c_fc'].shape

In [ ]:
ret[0]['transformer.h.0.mlp.c_proj'].shape

In [ ]:
class IKSAE(nn.Module):
    def __init__(self, hiden_dim=3072, vocab_size=100000, topk=4, emb_weight=None):
        super().__init__()
        self.encoder = nn.Linear(hiden_dim, vocab_size)
        self.decoder = nn.Linear(vocab_size, hiden_dim)
        self.decoder.weight = emb_weight
        self.k = topk
        
    
    def k_sparse(self, x):
        # 实现k-sparse约束
        topk, indices = torch.topk(x, self.k, dim=1)
        mask = torch.zeros_like(x).scatter_(1, indices, 1)
        return x * mask

    def forward(self, x):
        x = self.encoder(x)
        z = self.k_sparse(x)
        x_rec = self.decoder(z)
        return z, x_rec

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

In [ ]:
model_name = "gpt2"           # or "distilgpt2" for smaller
tokenizer = GPT2TokenizerFast.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained(model_name)
model.to("cuda")

In [ ]:
embed = model.transformer.wte

In [ ]:
vocab_size = embed.weight.size(0)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from torch.nn import functional as F

# 设置随机种子和设备
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 数据加载和预处理
transform = transforms.Compose([
    # 0-1 之间
    transforms.ToTensor(),
    # 这种运行不会出错的，数据/逻辑bug，很难排查
    # transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=102400, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=12800, shuffle=False)

def train_model(model, train_loader, num_epochs=20):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=100*1e-3)
    
    train_losses = []
    
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        
        for data, _ in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}'):
            data = data.to(device)
            optimizer.zero_grad()
            
            decoded, _ = model(data)
            loss = criterion(decoded, data.view(data.size(0), -1))
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        avg_loss = total_loss / len(train_loader)
        train_losses.append(avg_loss)
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.6f}')
    
    return train_losses

def visualize_results(model, test_loader, num_images=10, k=20):
    model.eval()
    with torch.no_grad():
        data = next(iter(test_loader))[0][:num_images].to(device)
        labels = next(iter(test_loader))[1][:num_images]
        decoded, encoded = model(data)
        
        # 可视化原始图像和重构图像
        fig, axes = plt.subplots(2, num_images, figsize=(20, 4))
        
        for i in range(num_images):
            # 原始图像
            err = F.mse_loss(data[i].cpu().squeeze(), decoded[i].cpu().view(28, 28))
            print(labels[i].item(), err.item())
            axes[0, i].imshow(data[i].cpu().squeeze(), cmap='gray')
            axes[0, i].axis('off')
            axes[0, i].set_title('Original')
            
            # 重构图像
            axes[1, i].imshow(decoded[i].cpu().view(28, 28), cmap='gray')
            axes[1, i].axis('off')
        
            axes[1, i].set_title(f'Reconstructed, {err.item():.2f}')
        
        plt.tight_layout()
        # plt.show()
        plt.savefig(f'./figs/ksae-reconstructed-{k}.png')
        
        # 可视化隐层激活
        plt.figure(figsize=(10, 4))
        plt.imshow(encoded.cpu().T, aspect='auto', cmap='viridis')
        plt.colorbar()
        plt.title('Hidden Layer Activations')
        plt.xlabel('Sample')
        plt.ylabel('Hidden Unit')
        # plt.show()
        plt.savefig(f'./figs/ksae-activations-{k}.png')

def plot_training_loss(losses, k):
    plt.figure(figsize=(10, 4))
    plt.plot(losses)
    plt.title(f'Training Loss Over Time (k={k})')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True)
    # plt.show()
    plt.savefig(f'./figs/ksae-loss-{k}.png')

# 主函数
def main():
    k = 20
    model = IKSAE(vocab_size=vocab_size, k = k, emb_weight=embed.weight).to(device)
    losses = train_model(model, train_loader)
    plot_training_loss(losses, k)
    visualize_results(model, test_loader, k=k)

